In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression



**Load dataset**

In [49]:
path = "/content/drive/MyDrive/Copy of homeprices.csv"

In [50]:
df = pd.read_csv(path)
df.head(5)

,area,bedrooms,age,price
0,2600,3.0,20,550000
1,3000,4.0,15,565000
2,3200,NaN,18,610000
3,3600,3.0,30,595000
4,4000,5.0,8,760000


***Data Preprocessing : Fill NA values with median value of a column***

In [51]:
df.bedrooms.median()

4.0

In [52]:
df.bedrooms = df.bedrooms.fillna(df.bedrooms.median())
df

,area,bedrooms,age,price
0,2600,3.0,20,550000
1,3000,4.0,15,565000
2,3200,4.0,18,610000
3,3600,3.0,30,595000
4,4000,5.0,8,760000
5,4100,6.0,8,810000


In [53]:
reg = LinearRegression()
# fit(input,output)
reg.fit(df.drop('price', axis='columns'), df.price)

LinearRegression()

In [54]:
reg.coef_   # weights of input features

array([  112.06244194, 23388.88007794, -3231.71790863])

In [55]:
reg.intercept_   # w0

np.float64(221323.00186540396)

In [56]:
reg.predict([[3000,4,15]])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([602590.07937407])

***Gradient descent***

In [57]:
def gradient_descent(X, y, learning_rate=0.05, iterations=20000):

    # extract row, column
    m, n = X.shape

    weights = np.zeros(n)
    bias = 0

    cost_history = []

    # Track the best parameters
    best_cost = float('inf')
    best_weights = weights.copy()
    best_bias = bias

    for i in range(iterations):

        # 1. Make predictions
        y_pred = X @ weights + bias

        # 2. Calculate cost
        cost = np.mean((y_pred - y) ** 2)
        cost_history.append(cost)

        # 3. Save parameters if cost is lower
        if cost < best_cost:
            best_cost = cost
            best_weights = weights.copy()
            best_bias = bias

        # 4. Calculate gradients
        dw = (2/m) * (X.T @ (y_pred - y))
        db = (2/m) * np.sum(y_pred - y)

        # 5. Update parameters
        weights = weights - learning_rate * dw
        bias = bias - learning_rate * db

    return best_weights, best_bias, cost_history


**Feature Scaling using StandardScaler**

In [61]:
from sklearn.preprocessing import StandardScaler
X = df.drop('price', axis='columns').values
y = df['price'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[-1.52162173 -1.09321633  0.46257302]
 [-0.77633762 -0.15617376 -0.19824558]
 [-0.40369556 -0.15617376  0.19824558]
 [ 0.34158855 -1.09321633  1.78421022]
 [ 1.08687266  0.78086881 -1.12339162]
 [ 1.27319369  1.71791138 -1.12339162]]


In [64]:
weights, bias, cost_history = gradient_descent(
    X_scaled,
    y=y,
    learning_rate=0.05,
    iterations=6000
    )

print(weights)
print(bias)
print(cost_history)

[ 60144.81724725  24960.31752077 -24452.38302269]
648333.3333333328
[np.float64(430258333333.3333), np.float64(346551051708.8994), np.float64(279623303359.6238), np.float64(225916296796.83316), np.float64(182703993631.43503), np.float64(147868787661.6201), np.float64(119747701620.01202), np.float64(97023929932.95874), np.float64(78648307102.98141), np.float64(63781082949.42483), np.float64(51747873305.21621), np.float64(42005779053.574326), np.float64(34116982201.604656), np.float64(27727959498.43056), np.float64(22552992627.434372), np.float64(18361012037.0006), np.float64(14965056303.375107), np.float64(12213801118.713247), np.float64(9984736386.849533), np.float64(8178661942.014485), np.float64(6715241910.741647), np.float64(5529411122.502294), np.float64(4568468532.670339), np.float64(3789725310.183985), np.float64(3158601157.6946254), np.float64(2647083098.4337406), np.float64(2232477516.0125012), np.float64(1896399531.9470222), np.float64(1623954514.855767), np.float64(1403075153

In [65]:
house = np.array([[3000, 4, 15]])

house_scaled = scaler.transform(house)

prediction_gd = house_scaled @ weights + bias

prediction_sklearn = reg.predict(
    pd.DataFrame([[3000, 4, 15]],
    columns=df.drop(columns='price').columns)
    )

print("Sklearn Prediction:", prediction_sklearn[0])
print("Gradient Descent Prediction:", prediction_gd[0])

Sklearn Prediction: 602590.07937407
Gradient Descent Prediction: 602590.0793761826
